Import de Pytorch et renommage du paquet nn qui contient des modules intéressants pour notre programme




In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F


Import de MulticlassAccuracy à partir de torcheval pour le calcul de la précision (accuracy) de notre modèle.

In [2]:
!pip install torcheval-nightly
from torcheval.metrics import MulticlassAccuracy


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.4/192.4 kB 2.8 MB/s eta 0:00:00


Montage de mon compte Google Drive. Vous pouvez retirer ces deux lignes si vous utilisez ce script en local sur votre machine

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive




```
# Ce texte est au format code
```

Définition de fonctions pour le chargement des fichiers de données

In [4]:
def load_data(train_file, test_file):
    X_train, Y_train = read_file(train_file)
    X_test, Y_test = read_file(test_file)

    return X_train, Y_train, X_test, Y_test

In [5]:
def read_file(filename: str):
    """
    Read a 'protein' file and return the lines
    :param filename:
    :return: lines
    """
    # read the file
    f = open(filename, "r")
    lines = f.readlines()
    print(lines)

    # remove the new line character
    lines = [line.strip() for line in lines]

    store_seq = []
    for idx, line in enumerate(lines):
        if line == "<>":
            store_seq.append([])
        elif line == "end" or line == "<end>":
            continue
        else:
            if line != "" and line[0] != "#":
                store_seq[-1].append(line)
    f.close()

    X, Y = [], []
    for seq in store_seq:
        x, y = [], []
        for line in seq:
            x.append(line[0])
            y.append(line[2])
        X.append(x)
        Y.append(y)

    return X, Y


Définition de la fonction qui crée un dictionnaire pour la correspondance entre symboles (acides aminés et structure secondaire) et indice. X et Y sont les listes renvoyées par la fonction load_data : X_train, Y_train ou X_test, Y_test.
Si vous avez quelques difficultés pour comprendre la ligne 8 par exemple (c'est du pur python) voici un peu de lecture : https://python.doctor/page-comprehension-list-listes-python-cours-debutants et https://groups.google.com/g/comp.lang.python/c/xvSFcnku46c



In [6]:
def create_amino_acids_dict(X, Y):
    """
    Create a dictionary with the amino acids and the secondary structure based on the training set
    :param X:
    :param Y:
    :return: dictionary with the amino acids and the secondary structure
    """
    acido_amino = set([x for x in X for x in x])

    # add special tokens
    acido_amino = ["<PAD>"] + list(acido_amino)
    acido_amino = {acido_amino[i]: i for i in range(0, len(acido_amino))}

    secondary_structure = set([x for x in Y for x in x])
    secondary_structure = list(secondary_structure)
    secondary_structure = {
        secondary_structure[i]: i for i in range(len(secondary_structure))
    }
    return acido_amino, secondary_structure


Fonction qui utilise les dictionnaires créés ci-dessus pour générer des listes de listes avec des indices. Renvoie une liste de liste dont chacune représente une protéine (= une séquence).
Gestion d'acide inconnu avec l'item "<UNK>"

In [8]:
def encode_data(X, Y, amino_acids_dict, secondary_structure_dict):
    """
    Encode the data
    :param X:
    :param Y:
    :param amino_acids_dict:
    :param secondary_structure_dict:
    :return:
    """
    X_encoded = []
    Y_encoded = []
    for x, y in zip(X, Y):
        x_encoded = []
        y_encoded = []
        for x_i, y_i in zip(x, y):
            try:
                x_encoded.append(amino_acids_dict[x_i])
            except:
                x_encoded.append(amino_acids_dict["<UNK>"])
            y_encoded.append(secondary_structure_dict[y_i])
        X_encoded.append(x_encoded)
        Y_encoded.append(y_encoded)
    return X_encoded, Y_encoded


Lectures des fichiers et préparation des données

In [9]:
train_path="/content/drive/My Drive/Enseignement/Approches_Neuronales_M1/Proteines/data/protein-secondary-structure.train"
test_path="/content/drive/My Drive/Enseignement/Approches_Neuronales_M1/Proteines/data/protein-secondary-structure.test"
X_train, Y_train, X_test, Y_test = load_data(train_path, test_path)


['# Training set for study of secondary structure of globular proteins by\n', '# Ning Qing and Terry Sejnowski.\n', '\n', '# Copyright (C) 1988 by Terrence J. Sejnowski.  Permission is hereby given to\n', '# use the included data for non-commercial research purposes.  Contact The\n', '# Johns Hopkins University, Cognitive Science Center, Baltimore MD, USA for\n', '# information on commercial use.\n', '\n', '<>\n', 'G _\n', 'V _\n', 'G _\n', 'T _\n', 'V _\n', 'P _\n', 'M _\n', 'T _\n', 'D _\n', 'Y _\n', 'G _\n', 'N _\n', 'D _\n', 'V _\n', 'E _\n', 'Y _\n', 'Y _\n', 'G _\n', 'Q _\n', 'V e\n', 'T e\n', 'I _\n', 'G _\n', 'T _\n', 'P _\n', 'G _\n', 'K _\n', 'S e\n', 'F e\n', 'N _\n', 'L _\n', 'N e\n', 'F e\n', 'D e\n', 'T _\n', 'G _\n', 'S _\n', 'S _\n', 'N _\n', 'L _\n', 'W e\n', 'V e\n', 'G _\n', 'S _\n', 'V _\n', 'Q _\n', 'C _\n', 'Q _\n', 'A _\n', 'S _\n', 'G _\n', 'C _\n', 'K _\n', 'G _\n', 'G _\n', 'R _\n', 'D _\n', 'K _\n', 'F _\n', 'N _\n', 'P _\n', 'S _\n', 'D _\n', 'G _\n', 'S _\n

In [10]:
amino_acids_dict, secondary_structure_dict = create_amino_acids_dict(
    X_train, Y_train
)

X_train_encoded, Y_train_encoded = encode_data(
    X_train, Y_train, amino_acids_dict, secondary_structure_dict
)
X_test_encoded, Y_test_encoded = encode_data(
    X_test, Y_test, amino_acids_dict, secondary_structure_dict
)

On retire les 10 dernières protéines pour construire un corpus de validation (ou développement). En effet, **le corpus de test ne doit jamais être utilisé durant la phase d'apprentissage ! **

In [11]:
# create a dev set
X_train_encoded, X_dev_encoded = X_train_encoded[:-10], X_train_encoded[-10:]
Y_train_encoded, Y_dev_encoded = Y_train_encoded[:-10], Y_train_encoded[-10:]


Définition de notre ProteinDataset qui hérite de Dataset

In [12]:
class ProteinDataset(torch.utils.data.Dataset):
    """Pytorch dataset for proteins."""

    def __init__(self, X, Y):
        self.X = X
        self.Y = Y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]


la fonction 'collate_fn' permet de construire les batches à partir des données. Elle est utilisée par la classe DataLoader. Il est parfois utile de créer sa propre fonction 'collate_fn'. Plus d'infos ici : https://pytorch.org/docs/stable/data.html#dataloader-collate-fn et  https://medium.com/geekculture/pytorch-datasets-dataloader-samplers-and-the-collat-fn-bbfc7c527cf1
Ici, 'batch' contient des listes de listes (la liste des fenêtres d'acides aminés et la liste des séquences d'étiquettes).


In [13]:
def custom_collate(batch):
    """
    Custom collate function
    :param batch:
    :return: padded sequences
    """
    # pad the sequences
    X = [torch.tensor(x[0]) for x in batch]
    Y = [torch.tensor(x[1]) for x in batch]

    # B, T, F
    X = torch.nn.utils.rnn.pad_sequence(X, batch_first=True, padding_value=0).unsqueeze(
        -1
    )
    Y = torch.nn.utils.rnn.pad_sequence(Y, batch_first=True, padding_value=-1)

    return X, Y


Python est un langage objet. Il est en général utile d'implémenter les modèles neuronaux sous forme de classe. Ce n'est pas obligatoire, mais cela facilite l'intégration dans de plus gros projet.
Ci-dessous, nous implémentons une classe RNN : la méthode _init appelé lors de la création d'une instance de la classe appelle 2 autres méthodes internes à la classe. Ceci n'est pas obligatoire. Notez toutefois l'initialisation des poids via l'initialisation de Xavier https://pytorch.org/docs/stable/nn.init.html#torch.nn.init.xavier_uniform_


In [14]:
class RNN(nn.Module):
    def __init__(
        self,
        input_size,
        output_size,
        emb_size=128,
        num_layers=2,
        hidden_size=256,
        rnn_type="RNN",
    ):
        """RNN model for protein classification.
        :param input_size: input size
        :param output_size: output size
        :param emb_size: embedding size
        :param num_layers: number of layers
        :param hidden_size: hidden size
        :param rnn_type: type of rnn to use
        """
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.input_size = input_size
        self.output_size = output_size
        self.emb = nn.Embedding(25, emb_size)

        self.rnn_type = rnn_type
        if self.rnn_type == "GRU":
            self.rnn = nn.GRU(
                emb_size,
                hidden_size,
                num_layers,
                batch_first=True,
                bidirectional=True,
                dropout=0.2,
            )
        elif self.rnn_type == "LSTM":
            self.rnn = nn.LSTM(
                emb_size,
                hidden_size,
                num_layers,
                batch_first=True,
                bidirectional=True,
                dropout=0.2,
            )
        else:
            self.rnn = nn.RNN(
                emb_size,
                hidden_size,
                num_layers,
                batch_first=True,
                bidirectional=True,
                dropout=0.2,
            )
        self.ln = nn.LayerNorm(2 * hidden_size)
        self.ln.bias.data.fill_(0)
        self.ln.bias.requires_grad = False
        self.fc = nn.Linear(2 * hidden_size, output_size)

    def forward(self, x, ht=None):
        """Forward pass.
        :param x: input
        :param ht: hidden state
        :return: output
        """
        if ht is None:
            # bidir
            ht = torch.zeros(self.num_layers * 2, x.size(0), self.hidden_size).to(
                x.device
            )
            if self.rnn_type == "LSTM":
                ct = torch.zeros(2 * self.num_layers, x.size(0), self.hidden_size).to(
                    x.device
                )

        # keep only the Batch and Time dimensions
        x = x.squeeze(-1)
        x = self.emb(x)
        x = x.view(x.size(0), x.size(1), -1)

        if self.rnn_type == "LSTM":
            out, (ht, ct) = self.rnn(x, (ht, ct))
        else:
            out, ht = self.rnn(x, ht)

        out = self.ln(out)
        out = F.dropout(out, p=0.2, training=self.training)
        out = self.fc(out)

        return out, ht #ATTENTION : ICI LA FONCTION RENVOIE UN TUPLE COMPOSE DE 2 TENSORS (en plus de la prédiction out, il y a l'état cachée courant ht)
        #note : on aurait pu se contenter de ne retourner que out, mais ht peut être utile dans certains circonstances (pas ici)

    def my_save(self, path):
      """ Save model for inference.
      """
      torch.save(self.state_dict(), path)



Création des datasets à partir des données préparées, puis création des dataloaders

In [15]:
batch_size=64

train_dataset = ProteinDataset(X_train_encoded, Y_train_encoded)
test_dataset = ProteinDataset(X_test_encoded, Y_test_encoded)
dev_dataset = ProteinDataset(X_dev_encoded, Y_dev_encoded)

train_dataloader = torch.utils.data.DataLoader(
train_dataset, batch_size=batch_size, shuffle=False, collate_fn=custom_collate
)
test_dataloader = torch.utils.data.DataLoader(
test_dataset, batch_size=batch_size, shuffle=False, collate_fn=custom_collate
)
dev_dataloader = torch.utils.data.DataLoader(
dev_dataset, batch_size=batch_size, shuffle=False, collate_fn=custom_collate
)

Création de l'instance de LSTM et affectation dans la variable model

In [16]:
inp_size = 1
out_size = 3
emb_size = 64
rnn_type = "LSTM"

num_layers=1
hidden_size= 64
device="cpu"

model = (
RNN(
inp_size,
emb_size,
num_layers=num_layers,
hidden_size=hidden_size,
)
.to(torch.float32)
.to(device)
)

/usr/local/lib/python3.11/dist-packages/torch/nn/modules/rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  warnings.warn(


Définition de l'optimiseur, ici AdamW, une variante d'Adam

In [17]:
lr=0.001
weight_decay=0.001

opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
loss_fn = torch.nn.CrossEntropyLoss(ignore_index=-1)

In [18]:
def accuracy_score(y_true, y_pred):
    y_pred = np.concatenate(tuple(y_pred))
    y_true = np.concatenate(tuple([[t for t in y] for y in y_true])).reshape(y_pred.shape)
    return (y_true == y_pred).sum() / float(len(y_true))

Ci-dessous l'apprentissage du réseau de neurones.
Notez:
- l'utilisation de tqdm qui permet de générer une bar de progression
- l'utilisation d'une métrique prédéfinie (MulticlassAccuracy, voir le second bloc de code de ce notebook pour l'import)

En ce qui concerne l'apprentissage :
- calcul de la loss
- mise à jour des poids via backward()
- mise à jour des paramètres de l'optimiseur via step()
- model.train() : met le modèle dans un état d'apprentissage
- model.eval() : met le modèle dans un état d'inférence (entre autre chose, neutralise le dropout)

In [19]:
from tqdm.auto import tqdm

nb_epochs=200
metric_dev_acc = MulticlassAccuracy()
best_dev_acc=0
best_epoch=0

for epoch in (bar := tqdm(range(nb_epochs))):
  model.train()
  train_loss = 0
  for x, y in train_dataloader:
    opt.zero_grad(set_to_none=True)
    pred,_=model.forward(x) # pred,_ car la fonction .forward(x) renvoie un tuple (voir commentaire dans la définition de la fonction)
                            # recupere la sortie du reseau de neurones dans "pred" ;
                            #comme nous n'avons pas besoin de ht, nous utilisons _ qui signifie que nous ignorons la seconde valeur du tuple renvoyée  par forward()
    pred=pred.permute(0, 2, 1) #Ceci est nécessaire car le modele ne renvoie pas les données sous la forme attendue par la fonction loss
                               #loss attend que pred soit sous la forme [batch, output] alors qu'avant permutation, pred est sous la forme [batch, seq, output]
                               #avec la permutation, pred devient sous la forme [batch, output, seq]
    loss = loss_fn(pred, y)
    train_loss += loss.item() / len(train_dataloader)
    loss.backward()
    opt.step()

  model.eval()
  dev_loss = 0
  for x, y in dev_dataloader:
    pred,_=model.forward(x)
    label_pred= torch.argmax(pred, dim=-1)
    pred=pred.permute(0, 2, 1)
    loss = loss_fn(pred, y)
    dev_loss += loss.item() / len(dev_dataloader)

    mask = y >= 0 # un padding a été utilisé pour construire des séquences de même longueur dans le batch. Nous avons mis la valeur -1 à l'étiquette dans ce cas
                  # avec la ligne ci-dessus on construit un tensor qui contient "True" pour les éléments de la séquence d'origine et "False" pour les éléments ajoutés
    y_without_padding=torch.masked_select(y, mask) #permet de construire un tensor à partir de y sans les éléments "padded" voir https://pytorch.org/docs/stable/generated/torch.masked_select.html
    label_pred_without_padding=torch.masked_select(label_pred, mask) # idem pour les prédictions
    metric_dev_acc.update(label_pred_without_padding,y_without_padding) # ainsi nous pouvons évaluer sans interférénces des éléments 'padded'

  dev_acc = metric_dev_acc.compute()

  if best_dev_acc<dev_acc:
    best_dev_acc=dev_acc
    best_epoch=epoch
    model.my_save('/content/drive/My Drive/Enseignement/Approches_Neuronales_M1/Proteines/best_model_LSTM.pt')
    bar.set_description(
              f"Epoch: {epoch} - Train Loss {train_loss:.4f} - Dev Loss {dev_loss:.4f} - Dev Acc {dev_acc:.4f} - Best Dev Acc {best_dev_acc:.4f}"
          )
    bar.update()

  else:
      bar.set_description(
              f"Epoch: {epoch} - Train Loss {train_loss:.4f} - Dev Loss {dev_loss:.4f} - Dev Acc {dev_acc:.4f} - Best Dev Acc {best_dev_acc:.4f}"
          )
      bar.update()

print("Best model at epoch", epoch, "with an accuracy of", float(best_dev_acc), "on the dev dataset")




  0%|          | 0/200 [00:00<?, ?it/s]

Best model at epoch 199 with an accuracy of 0.5970474481582642 on the dev dataset


Evaluation des performances du modèle en terme de précision sur le **corpus de test**

In [20]:
test_loss = 0
metric_test_acc = MulticlassAccuracy()
best_model = (
RNN(
inp_size,
emb_size,
num_layers=num_layers,
hidden_size=hidden_size,
)
.to(torch.float32)
.to(device)
)


best_model.load_state_dict(torch.load('/content/drive/My Drive/Enseignement/Approches_Neuronales_M1/Proteines/best_model_LSTM.pt'))

print(best_model)
best_model.eval()
for x, y in test_dataloader:
    pred,_=best_model.forward(x)
    label_pred= torch.argmax(pred, dim=-1)
    pred=pred.permute(0, 2, 1)
    loss = loss_fn(pred, y)
    test_loss += loss.item() / len(test_dataloader)
    mask = y >= 0
    y_without_padding=torch.masked_select(y, mask)
    label_pred_without_padding=torch.masked_select(label_pred, mask)
    metric_test_acc.update(label_pred_without_padding,y_without_padding)


test_acc = metric_test_acc.compute()

print("Accuracy Test: ", test_acc)

RNN(
  (emb): Embedding(25, 128)
  (rnn): RNN(128, 64, batch_first=True, dropout=0.2, bidirectional=True)
  (ln): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
  (fc): Linear(in_features=128, out_features=64, bias=True)
)
Accuracy Test:  tensor(0.6085)


/usr/local/lib/python3.11/dist-packages/torch/nn/modules/rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  warnings.warn(
